# Stage 00_gap_exploration: gap_exploration
Configuration, inputs and results remain in the corresponding stages directory.


### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
gap_exploration_experiment_dir = ROOT / 'experiment/research/gap_mechanisms/stages/00_gap_exploration'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

gap_exploration_data_paths = ExperimentPaths(ROOT, gap_exploration_experiment_dir / 'config/paths.json')


# BPOSD–Tesseract gap: stopping, handoff, and selective OSD

All decoders use paired physical shots. A decoder succeeds only when its predicted logical sector equals the sampled truth; syndrome validity is recorded separately for correction-producing decoders.

In [ ]:
import json, sys
from dataclasses import dataclass
from math import comb
from pathlib import Path
from time import perf_counter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from tqdm.auto import tqdm
from beliefmatching import detector_error_model_to_check_matrices
from codes.ca_code import CA_9_2_5, TriangularCode
from decoders.cpp_backend import Tesseract
from decoders.cython import BPOSD
from decoders.cython.cython_core.osd_core import ordered_statistics_postprocess
from noises_ca import CircuitBuilder, ErrorRate

gap_exploration_config = json.loads((gap_exploration_experiment_dir / 'config/bposd_tesseract_gap_next_experiments.json').read_text(encoding='utf-8'))
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.
# reuse_existing_data: boolean (True or False).
# levels: non-empty; each item is one of {1, 2, 3}.
# p_values: each item is in (0, 1).
# shots_per_point: positive.
# Expected relationship: bp_max_iterations >= maximum of snapshot_iterations.
# Expected relationship: long_bp_iterations >= bp_max_iterations.
# osd_orders_by_level: include a string key for every entry in levels.
# values of osd_orders_by_level: each order list is sorted, unique, and begins with [0, 1].
# development_fraction: in (0, 1).
# cv_folds: at least 2.
# patience_values, top_k_values, iteration_values: each item is positive.
# damping_values: each item is in [0, 1).
# scaling_values: each item is positive.
# trigger_quantiles: each item is in [0, 1].
# schedule_values: values restricted to {'parallel', 'serial'}.
# snapshot_cases_per_stratum: positive.
# trajectory_cases_per_stratum: positive.
# handoff_grid_cases: positive.
# tesseract_beam: positive.
# tesseract_pq_limit: positive.

if gap_exploration_config['smoke_mode']:
    gap_exploration_config.update(
        levels=[1],
        p_values=[0.1],
        shots_per_point=200,
        handoff_grid_cases=10,
        cv_folds=2,
    )
gap_exploration_out = gap_exploration_data_paths.run(smoke_mode=gap_exploration_config['smoke_mode'])

from decoders.cython.cython_core.osd_core import OSD_VERSION, osd_candidate_count
from experiment.utils.misc import check_algorithm_cache
check_algorithm_cache(gap_exploration_data_paths, gap_exploration_config, {'osd': OSD_VERSION})

from experiment.utils.bp_trace import check_bp_cache
check_bp_cache(gap_exploration_data_paths, gap_exploration_config)

gap_exploration_stats = gap_exploration_data_paths.output('stats')
gap_exploration_trace_dir = gap_exploration_data_paths.output('trace')
gap_exploration_conf = gap_exploration_data_paths.output('config')

for gap_exploration_path in (gap_exploration_stats, gap_exploration_trace_dir, gap_exploration_conf):
    gap_exploration_path.mkdir(parents=True, exist_ok=True)
gap_exploration_data_paths.output('config_json').write_text(json.dumps(gap_exploration_config, indent=2), encoding='utf-8')

from experiment.utils import misc

def gap_exploration_log(level, payload):
    return misc.log(level, payload, CONFIG=gap_exploration_config)

sns.set_theme(style='whitegrid')
gap_exploration_log(2, {'stats': str(gap_exploration_stats), 'trace': str(gap_exploration_trace_dir)})


In [ ]:
from experiment.utils.bp_trace import BPContext

@dataclass(frozen=True)
class Context(BPContext):
    distance: int
    level: int
    p: float
    H: np.ndarray
    L: np.ndarray
    priors: np.ndarray
    circuit: object
    dem: object

@dataclass(frozen=True)
class gap_exploration_BPTrace:
    llr: np.ndarray
    hard: np.ndarray
    valid: np.ndarray
    runtime_seconds: np.ndarray
    first_valid: int | None

CACHE = {}

def gap_exploration_gf2_rank(matrix):
    A = np.asarray(matrix, dtype=np.uint8).copy() % 2
    rank = 0
    for column in range(A.shape[1]):
        pivot = next((row for row in range(rank, A.shape[0]) if A[row, column]), None)
        if pivot is None:
            continue
        A[[rank, pivot]] = A[[pivot, rank]]

        for row in range(A.shape[0]):
            if row != rank and A[row, column]:
                A[row] ^= A[rank]

        rank += 1

    return rank

def gap_exploration_context_for(level, p):
    key = (int(level), float(p))
    if key in CACHE:
        return CACHE[key]
    code = CA_9_2_5() if level == 2 else TriangularCode(level)
    circuit = CircuitBuilder(code).get_circuit_level_circuit(ErrorRate(p=float(p), p_meas=0), d_rounds=1)
    dem = circuit.detector_error_model()
    m = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
    H = m.check_matrix.toarray().astype(np.uint8) % 2
    L = m.observables_matrix.toarray().astype(np.uint8) % 2
    priors = np.asarray(m.priors, float)
    assert np.all(H.sum(axis=1) >= 2) and np.all((priors > 0) & (priors < 1)) and (gap_exploration_gf2_rank(H) == H.shape[0])
    distance = 5 if level == 2 else int(code.d)
    CACHE[key] = Context(distance, level, float(p), H, L, priors, circuit, dem)

    return CACHE[key]

def orders_for(context):
    return gap_exploration_config['osd_orders_by_level'][str(context.level)]

def primary_order_for(context):
    return orders_for(context)[-1]

def bp_trace(
    context,
    syndrome,
    iterations,
    scaling=1.0,
    damping=0.0,
    schedule='parallel',
):
    trace = context.bp_decoder.trace(
        syndrome, iterations, scaling=scaling, damping=damping, schedule=schedule,
        accumulation='sum_then_prior', serial_accumulation='sum_then_prior',
        record_messages=False,
    )
    return gap_exploration_BPTrace(trace.llr, trace.hard, trace.valid, trace.runtime_seconds, trace.first_valid)

def osd_snapshot(
    context,
    syndrome,
    llr,
    order,
):
    return ordered_statistics_postprocess(np.ascontiguousarray(llr, dtype=np.float64), np.ascontiguousarray(syndrome, dtype=np.uint8), np.ascontiguousarray(context.H, dtype=np.uint8), int(order), np.ascontiguousarray(context.priors, dtype=np.float64))

def logical_correct(context, correction, truth):
    return bool(np.array_equal(context.L @ correction % 2, truth))

def syndrome_consistent(context, correction, syndrome):
    return bool(np.array_equal(context.H @ correction % 2, syndrome))

def gap_exploration_prior_cost(context, correction):
    e = np.asarray(correction, bool)
    return float((-np.log(np.where(e, context.priors, 1 - context.priors))).sum())

def gap_exploration_bits(values):
    return ''.join(map(str, np.asarray(values, dtype=np.uint8).tolist()))

def wilson_interval(errors, shots, z=1.959963984540054):
    assert 0 <= errors <= shots and shots > 0
    p = errors / shots
    denominator = 1 + z * z / shots
    center = (p + z * z / (2 * shots)) / denominator
    half = z * np.sqrt(p * (1 - p) / shots + z * z / (4 * shots * shots)) / denominator

    return (center - half, center + half)

def unpack_predictions(packed, count):
    return np.unpackbits(packed, axis=1, count=count, bitorder='little').astype(np.uint8)

def read_experiment_csv(path):
    assert path.is_file(), f'missing cache file: {path}'
    return pd.read_csv(path, dtype={'syndrome': str, 'true_logical': str}, low_memory=False)

def requested_distances():
    return {5 if level == 2 else int(TriangularCode(level).d) for level in gap_exploration_config['levels']}

def filter_distances(frame):
    return frame[frame.distance.isin(requested_distances())].copy() if 'distance' in frame.columns else frame.copy()

def gap_exploration_bit_array(value, length):
    text = str(value).strip().zfill(length)
    assert len(text) == length and set(text) <= {'0', '1'}
    return np.fromiter((int(bit) for bit in text), dtype=np.uint8, count=length)

def case_from_row(row, iterations=None):
    context = gap_exploration_context_for(int(row.level), float(row.p))
    syndrome = gap_exploration_bit_array(row.syndrome, context.H.shape[0])
    truth = gap_exploration_bit_array(row.true_logical, context.L.shape[0])
    trace = bp_trace(context, syndrome, gap_exploration_config['long_bp_iterations'] if iterations is None else int(iterations))

    return (context, syndrome, truth, trace)

def gap_class(b, f, t):
    if b and f and t:
        return 'shared_success'
    if not b and f and t:
        return 'bposd_early_stop_gap'

    if not b and (not f) and t:
        return 'forced_osd_still_below_tesseract'
    if not b and (not f) and (not t):
        return 'shared_hard_case'

    return 'unusual_disagreement'


## Experiments 1–5: paired gap, OSD order, snapshots, ranking, and BP oracle

Decode the same physical shots with the reference methods, decompose their disagreements and test OSD at different orders and BP snapshots. This first establishes whether the sampled regime contains the target gap events before interpreting a handoff mechanism.


In [ ]:
pair_paths = [gap_exploration_data_paths.output('gap_decomposition_csv'), gap_exploration_data_paths.output('reliability_snapshots_csv'), gap_exploration_data_paths.output('bp_valid_candidates_csv'), gap_exploration_data_paths.output('forced_osd_order_trials_csv')]
existing_pair_files = sum((path.is_file() for path in pair_paths))
assert existing_pair_files in {0, len(pair_paths)}, 'paired cache is partial; restore or remove the incomplete stage files'
pair_cache_ready = gap_exploration_config['reuse_existing_data'] and existing_pair_files == len(pair_paths)
CASE_STORE = {}

if pair_cache_ready:
    shots = filter_distances(read_experiment_csv(pair_paths[0]))
    snapshots = filter_distances(read_experiment_csv(pair_paths[1]))
    order_trials = filter_distances(read_experiment_csv(pair_paths[3]))
    valid_states = read_experiment_csv(pair_paths[2])
    valid_states = valid_states[valid_states.shot_id.isin(shots.shot_id)].copy()
    assert len(shots)
    gap_exploration_log(2, {
        'stage': 'paired_population',
        'source': 'cache',
        'shots': len(shots),
        'distances': sorted(shots.distance.unique()),
    })

shot_rows = []
snapshot_rows = []
valid_rows = []
order_trial_rows = []
gap_exploration_shot_id = 0

for gap_exploration_level in [] if pair_cache_ready else gap_exploration_config['levels']:
    for p_index, gap_exploration_p in enumerate(gap_exploration_config['p_values']):
        gap_exploration_context = gap_exploration_context_for(gap_exploration_level, gap_exploration_p)
        gap_exploration_sampler = gap_exploration_context.dem.compile_sampler(seed=gap_exploration_config['random_seed'] + gap_exploration_level * 1009 + p_index)
        gap_exploration_syndromes, gap_exploration_truths, gap_exploration_errors = gap_exploration_sampler.sample(shots=gap_exploration_config['shots_per_point'], return_errors=True)
        gap_exploration_tess = Tesseract(beam=gap_exploration_config['tesseract_beam'], pq_limit=gap_exploration_config['tesseract_pq_limit'], seed=gap_exploration_config['random_seed']).compile_decoder_for_dem(dem=gap_exploration_context.dem)
        tess_started = perf_counter()
        tess_packed = gap_exploration_tess.decode_shots_bit_packed(bit_packed_detection_event_data=np.packbits(gap_exploration_syndromes, axis=1, bitorder='little'))
        tess_per_shot = (perf_counter() - tess_started) / len(gap_exploration_syndromes)
        tess_predictions = unpack_predictions(tess_packed, gap_exploration_context.dem.num_observables)
        primary_order = primary_order_for(gap_exploration_context)
        gap_exploration_standard = BPOSD(max_iter=gap_exploration_config['bp_max_iterations'], osd_order=primary_order)
        gap_exploration_standard._init_backend(dem=gap_exploration_context.dem)
        gap_exploration_forced = BPOSD(max_iter=gap_exploration_config['bp_max_iterations'], osd_order=primary_order)
        gap_exploration_forced._init_backend(dem=gap_exploration_context.dem)

        for gap_exploration_local, (gap_exploration_syndrome, gap_exploration_truth, gap_exploration_error, tess_pred) in enumerate(tqdm(zip(gap_exploration_syndromes, gap_exploration_truths, gap_exploration_errors, tess_predictions), total=len(gap_exploration_syndromes), desc=f'paired d={gap_exploration_context.distance} p={gap_exploration_p:g}', disable=gap_exploration_config['print_level'] == 1)):
            gap_exploration_syndrome = np.asarray(gap_exploration_syndrome, np.uint8)
            gap_exploration_truth = np.asarray(gap_exploration_truth, np.uint8)
            gap_exploration_error = np.asarray(gap_exploration_error, np.uint8)
            split = 'development' if (gap_exploration_shot_id * 2654435761 + gap_exploration_config['random_seed']) % 10000 < 10000 * gap_exploration_config['development_fraction'] else 'test'
            gap_exploration_trace = bp_trace(gap_exploration_context, gap_exploration_syndrome, gap_exploration_config['long_bp_iterations'])
            bp_runtime = float(gap_exploration_trace.runtime_seconds[-1])
            handoff = gap_exploration_trace.first_valid if gap_exploration_trace.first_valid is not None and gap_exploration_trace.first_valid <= gap_exploration_config['bp_max_iterations'] else gap_exploration_config['bp_max_iterations']
            started = perf_counter()
            std_corr = gap_exploration_standard.decode_to_correction(gap_exploration_syndrome, force_osd=False)
            std_runtime = perf_counter() - started
            started = perf_counter()
            forced_corr = gap_exploration_forced.decode_to_correction(gap_exploration_syndrome, force_osd=True)
            gap_exploration_forced_runtime = perf_counter() - started
            plain_corr = gap_exploration_trace.hard[handoff]
            gap_exploration_bp_valid = syndrome_consistent(gap_exploration_context, plain_corr, gap_exploration_syndrome)
            std_ok = logical_correct(gap_exploration_context, std_corr, gap_exploration_truth)
            forced_ok = logical_correct(gap_exploration_context, forced_corr, gap_exploration_truth)
            tess_ok = bool(np.array_equal(tess_pred, gap_exploration_truth))
            plain_ok = logical_correct(gap_exploration_context, plain_corr, gap_exploration_truth)
            order_corr = {}
            order_runtime = {}
            information_bits = gap_exploration_context.H.shape[1] - gap_exploration_context.H.shape[0]

            for q in orders_for(gap_exploration_context):
                osd_started = perf_counter()
                order_corr[q] = osd_snapshot(gap_exploration_context, gap_exploration_syndrome, gap_exploration_trace.llr[handoff], q)
                order_runtime[q] = perf_counter() - osd_started
                order_trial_rows.append({
                    'shot_id': gap_exploration_shot_id,
                    'distance': gap_exploration_context.distance,
                    'p': gap_exploration_p,
                    'split': split,
                    'osd_order': q,
                    'logical_correct': int(logical_correct(gap_exploration_context, order_corr[q], gap_exploration_truth)),
                    'runtime_seconds': order_runtime[q],
                    'candidate_count': osd_candidate_count(information_bits, q),
                })

            abs_llr = np.abs(gap_exploration_trace.llr[handoff])
            sorted_abs = np.sort(abs_llr)
            recent = gap_exploration_trace.hard[max(0, handoff - 8):handoff + 1]
            recent_valid = [tuple(x.tolist()) for x, v in zip(recent, gap_exploration_trace.valid[max(0, handoff - 8):handoff + 1]) if v]
            logical_recent = [gap_exploration_bits(gap_exploration_context.L @ np.asarray(x, np.uint8) % 2) for x in recent_valid]
            gap_exploration_instability = float(np.mean(np.abs(np.diff(gap_exploration_trace.llr[max(0, handoff - 4):handoff + 1], axis=0)))) if handoff else 0.0
            instability_damping = 0.0
            instability_branch_evaluated = int(not gap_exploration_bp_valid)
            instability_corr = forced_corr
            instability_branch_runtime = 0.0

            if not gap_exploration_bp_valid:
                gap_exploration_scale = gap_exploration_instability / (float(np.median(abs_llr)) + 1e-12)
                instability_damping = float(np.clip(gap_exploration_scale / (1 + gap_exploration_scale), 0, 0.5))
                refined_trace = bp_trace(gap_exploration_context, gap_exploration_syndrome, gap_exploration_config['long_bp_iterations'], damping=instability_damping)
                refined_handoff = refined_trace.first_valid if refined_trace.first_valid is not None else gap_exploration_config['long_bp_iterations']
                instability_started = perf_counter()
                instability_corr = osd_snapshot(gap_exploration_context, gap_exploration_syndrome, refined_trace.llr[refined_handoff], primary_order)
                instability_branch_runtime = float(gap_exploration_trace.runtime_seconds[gap_exploration_config['bp_max_iterations']] + refined_trace.runtime_seconds[refined_handoff] + perf_counter() - instability_started)

            gap_exploration_row = {
                'shot_id': gap_exploration_shot_id,
                'distance': gap_exploration_context.distance,
                'level': gap_exploration_level,
                'p': gap_exploration_p,
                'split': split,
                'syndrome': gap_exploration_bits(gap_exploration_syndrome),
                'true_logical': gap_exploration_bits(gap_exploration_truth),
                'bp_syndrome_consistent': int(gap_exploration_bp_valid),
                'bp_logical_correct': int(plain_ok),
                'bposd_logical_correct': int(std_ok),
                'forced_osd_logical_correct': int(forced_ok),
                'tesseract_logical_correct': int(tess_ok),
                'gap_class': gap_class(std_ok, forced_ok, tess_ok),
                'bp_iterations': handoff,
                'first_valid_iteration': gap_exploration_trace.first_valid,
                'bp_runtime_seconds': bp_runtime,
                'plain_runtime_seconds': float(gap_exploration_trace.runtime_seconds[handoff]),
                'standard_runtime_seconds': std_runtime,
                'forced_runtime_seconds': gap_exploration_forced_runtime,
                'tesseract_runtime_seconds': tess_per_shot,
                'llr_min': float(abs_llr.min()),
                'llr_q10': float(np.quantile(abs_llr, 0.1)),
                'llr_q25': float(np.quantile(abs_llr, 0.25)),
                'llr_median': float(np.median(abs_llr)),
                'llr_gap_5_6': float(sorted_abs[min(5, len(sorted_abs) - 1)] - sorted_abs[min(4, len(sorted_abs) - 1)]),
                'recent_valid_count': len(recent_valid),
                'recent_unique_valid': len(set(recent_valid)),
                'recent_logical_switches': sum((a != b for a, b in zip(logical_recent, logical_recent[1:]))),
                'belief_instability': gap_exploration_instability,
                'false_stop': int(gap_exploration_bp_valid and (not std_ok) and forced_ok),
                'gap_target': int(not std_ok and tess_ok),
                'osd0_correct': int(logical_correct(gap_exploration_context, order_corr[0], gap_exploration_truth)),
                'osd1_correct': int(logical_correct(gap_exploration_context, order_corr[1], gap_exploration_truth)),
                'osd0_runtime_seconds': order_runtime[0],
                'osd1_runtime_seconds': order_runtime[1],
                'osdcs_runtime_seconds': order_runtime[primary_order],
                'osdcs_correct': int(logical_correct(gap_exploration_context, order_corr[primary_order], gap_exploration_truth)),
                'osd0_disagrees': int(not np.array_equal(order_corr[0], plain_corr)),
                'osd0_logical_disagrees': int(not np.array_equal(gap_exploration_context.L @ order_corr[0] % 2, gap_exploration_context.L @ plain_corr % 2)),
                'osd0_cost_gain': gap_exploration_prior_cost(gap_exploration_context, plain_corr) - gap_exploration_prior_cost(gap_exploration_context, order_corr[0]),
                'prior_osd_correct': int(logical_correct(gap_exploration_context, osd_snapshot(gap_exploration_context, gap_exploration_syndrome, np.log((1 - gap_exploration_context.priors) / gap_exploration_context.priors), primary_order), gap_exploration_truth)),
                'early_osd_correct': int(logical_correct(gap_exploration_context, osd_snapshot(gap_exploration_context, gap_exploration_syndrome, gap_exploration_trace.llr[min(8, len(gap_exploration_trace.llr) - 1)], primary_order), gap_exploration_truth)),
                'instability_osd_correct': int(logical_correct(gap_exploration_context, instability_corr, gap_exploration_truth)),
                'instability_damping': instability_damping,
                'instability_branch_evaluated': instability_branch_evaluated,
                'instability_branch_runtime_seconds': instability_branch_runtime,
            }

            shot_rows.append(gap_exploration_row)
            CASE_STORE[gap_exploration_shot_id] = (gap_exploration_context, gap_exploration_syndrome, gap_exploration_truth, gap_exploration_error, gap_exploration_trace, std_corr, forced_corr, order_corr)
            error_support = np.flatnonzero(gap_exploration_error)
            ranks = rankdata(abs_llr, method='average')

            if gap_exploration_local < gap_exploration_config['snapshot_cases_per_stratum']:
                for gap_exploration_t in gap_exploration_config['snapshot_iterations']:
                    if gap_exploration_t > gap_exploration_config['long_bp_iterations']:
                        continue
                    corr = osd_snapshot(gap_exploration_context, gap_exploration_syndrome, gap_exploration_trace.llr[gap_exploration_t], primary_order)
                    gap_exploration_support = np.flatnonzero(gap_exploration_error)
                    rank_t = rankdata(np.abs(gap_exploration_trace.llr[gap_exploration_t]), method='average')
                    snapshot_rows.append({
                        'shot_id': gap_exploration_shot_id,
                        'distance': gap_exploration_context.distance,
                        'p': gap_exploration_p,
                        'split': split,
                        'iteration': gap_exploration_t,
                        'relative_iteration': np.nan if gap_exploration_trace.first_valid is None else gap_exploration_t - gap_exploration_trace.first_valid,
                        'osd_logical_correct': int(logical_correct(gap_exploration_context, corr, gap_exploration_truth)),
                        'bp_syndrome_consistent': int(gap_exploration_trace.valid[gap_exploration_t]),
                        'bp_logical_correct': int(logical_correct(gap_exploration_context, gap_exploration_trace.hard[gap_exploration_t], gap_exploration_truth)),
                        'mean_error_variable_rank': float(rank_t[gap_exploration_support].mean()) if len(gap_exploration_support) else np.nan,
                        **{f'error_top_{k}_fraction': float(np.isin(gap_exploration_support, np.argsort(np.abs(gap_exploration_trace.llr[gap_exploration_t]))[:min(k, len(rank_t))]).mean()) if len(gap_exploration_support) else np.nan for k in gap_exploration_config['top_k_values']},
                    })

            if gap_exploration_local < gap_exploration_config['trajectory_cases_per_stratum']:
                gap_exploration_seen = set()
                for gap_exploration_t, (gap_exploration_hard, is_valid) in enumerate(zip(gap_exploration_trace.hard, gap_exploration_trace.valid)):
                    gap_exploration_key = gap_exploration_hard.tobytes()
                    if is_valid and gap_exploration_key not in gap_exploration_seen:
                        gap_exploration_seen.add(gap_exploration_key)
                        valid_rows.append({
                            'shot_id': gap_exploration_shot_id,
                            'iteration': gap_exploration_t,
                            'logical_sector': gap_exploration_bits(gap_exploration_context.L @ gap_exploration_hard % 2),
                            'logical_correct': int(logical_correct(gap_exploration_context, gap_exploration_hard, gap_exploration_truth)),
                            'prior_cost': gap_exploration_prior_cost(gap_exploration_context, gap_exploration_hard),
                        })

            gap_exploration_shot_id += 1

if not pair_cache_ready:
    shots = pd.DataFrame(shot_rows)
    snapshots = pd.DataFrame(snapshot_rows)
    valid_states = pd.DataFrame(valid_rows)
    order_trials = pd.DataFrame(order_trial_rows)

    shots.to_csv(pair_paths[0], index=False)
    snapshots.to_csv(pair_paths[1], index=False)
    valid_states.to_csv(pair_paths[2], index=False)
    order_trials.to_csv(pair_paths[3], index=False)
    gap_exploration_log(2, {
        'stage': 'paired_population',
        'source': 'experiment',
        'shots': len(shots),
        'distances': sorted(shots.distance.unique()),
    })


In [ ]:
gap_summary = shots.groupby(['distance', 'p', 'gap_class'], as_index=False).size().rename(columns={'size': 'shots'})
gap_metrics = []
for (gap_exploration_distance, gap_exploration_p), g in shots.groupby(['distance', 'p']):
    gap_exploration_denominator = np.sum(g.bposd_logical_correct.eq(0) & g.tesseract_logical_correct.eq(1))
    numerator = np.sum(g.gap_class.eq('bposd_early_stop_gap'))
    gap_metrics.append({
        'distance': gap_exploration_distance,
        'p': gap_exploration_p,
        'shots': len(g),
        'eta_stop': numerator / gap_exploration_denominator if gap_exploration_denominator else np.nan,
        'eta_forced_osd': g.loc[g.bposd_logical_correct.eq(0), 'forced_osd_logical_correct'].mean(),
        'bp_valid_given_gap': g.loc[g.gap_target.eq(1), 'bp_syndrome_consistent'].mean(),
    })

gap_metrics = pd.DataFrame(gap_metrics)
gap_summary.to_csv(gap_exploration_data_paths.output('gap_class_summary_csv'), index=False)
gap_metrics.to_csv(gap_exploration_data_paths.output('gap_closure_fraction_csv'), index=False)
decoder_ler = []

for (gap_exploration_distance, gap_exploration_p), g in shots.groupby(['distance', 'p']):
    for decoder, gap_exploration_column in {
        'standard_BPOSD': 'bposd_logical_correct',
        'forced_OSD_BPOSD': 'forced_osd_logical_correct',
        'Tesseract': 'tesseract_logical_correct',
    }.items():
        gap_exploration_errors = int((1 - g[gap_exploration_column]).sum())
        gap_exploration_low, high = wilson_interval(gap_exploration_errors, len(g))
        decoder_ler.append({
            'distance': gap_exploration_distance,
            'p': gap_exploration_p,
            'decoder': decoder,
            'shots': len(g),
            'logical_errors': gap_exploration_errors,
            'LER': gap_exploration_errors / len(g),
            'LER_CI95_low': gap_exploration_low,
            'LER_CI95_high': high,
        })

decoder_ler = pd.DataFrame(decoder_ler)
decoder_ler.to_csv(gap_exploration_data_paths.output('gap_decoder_ler_csv'), index=False)
order_analysis = order_trials.merge(shots[['shot_id', 'bp_syndrome_consistent', 'bposd_logical_correct', 'tesseract_logical_correct']], on='shot_id', how='left')
primary_order_subset = order_analysis[order_analysis.bposd_logical_correct.eq(0) & order_analysis.bp_syndrome_consistent.eq(1) & order_analysis.tesseract_logical_correct.eq(1)]
order_analysis = pd.concat([order_analysis.assign(subset='full_population'), primary_order_subset.assign(subset='syndrome_consistent_bposd_wrong_tesseract_correct')], ignore_index=True)
order_summary = order_analysis.groupby(['distance', 'p', 'subset', 'osd_order'], as_index=False).agg(
    shots=('shot_id', 'size'),
    success_rate=('logical_correct', 'mean'),
    mean_runtime_seconds=('runtime_seconds', 'mean'),
    average_candidate_count=('candidate_count', 'mean'),
).assign(average_osd_calls=1.0)
order_gains = []

for (gap_exploration_distance, gap_exploration_p, gap_exploration_subset), gap_exploration_group in order_summary.groupby(['distance', 'p', 'subset']):
    rates = gap_exploration_group.set_index('osd_order').success_rate
    order_gains.append({
        'distance': gap_exploration_distance,
        'p': gap_exploration_p,
        'subset': gap_exploration_subset,
        'delta_R1': rates.loc[1] - rates.loc[0],
        'delta_RCS': rates.iloc[-1] - rates.loc[1],
    })

order_summary = order_summary.merge(pd.DataFrame(order_gains), on=['distance', 'p', 'subset'], how='left')
order_summary.to_csv(gap_exploration_data_paths.output('forced_osd_order_ablation_csv'), index=False)
snapshot_summary = snapshots.groupby(['distance', 'p', 'iteration'], as_index=False).agg(osd_success_rate=('osd_logical_correct', 'mean'), bp_success_rate=('bp_logical_correct', 'mean'), mean_error_rank=('mean_error_variable_rank', 'mean'))
relative_snapshot = snapshots.dropna(subset=['relative_iteration']).groupby('relative_iteration', as_index=False).agg(osd_success_rate=('osd_logical_correct', 'mean'), shots=('shot_id', 'size'))

snapshot_summary.to_csv(gap_exploration_data_paths.output('reliability_snapshot_summary_csv'), index=False)
relative_snapshot.to_csv(gap_exploration_data_paths.output('relative_snapshot_summary_csv'), index=False)
rank_groups = shots[['shot_id', 'bp_logical_correct', 'forced_osd_logical_correct']].assign(group=lambda x: np.select([x.bp_logical_correct.eq(1), x.forced_osd_logical_correct.eq(1)], ['BP correct', 'BP wrong / ForcedOSD correct'], default='BP wrong / ForcedOSD wrong'))
ranking_quality = snapshots.merge(rank_groups[['shot_id', 'group']], on='shot_id').groupby(['group', 'iteration'], as_index=False).agg(mean_error_rank=('mean_error_variable_rank', 'mean'), **{f'mean_top_{k}_coverage': (f'error_top_{k}_fraction', 'mean') for k in gap_exploration_config['top_k_values']})

ranking_quality.to_csv(gap_exploration_data_paths.output('reliability_ranking_quality_csv'), index=False)
gap_exploration_oracle = valid_states.groupby('shot_id', as_index=False).agg(valid_candidates=('logical_sector', 'size'), logical_sectors=('logical_sector', 'nunique'), trajectory_oracle=('logical_correct', 'max'))
gap_exploration_oracle = shots[['shot_id', 'distance', 'p', 'bp_logical_correct']].merge(gap_exploration_oracle, on='shot_id', how='left').fillna({'valid_candidates': 0, 'logical_sectors': 0, 'trajectory_oracle': 0})
oracle_summary = gap_exploration_oracle.groupby(['distance', 'p'], as_index=False).agg(oracle_success_rate=('trajectory_oracle', 'mean'), mean_valid_candidates=('valid_candidates', 'mean'), mean_logical_sectors=('logical_sectors', 'mean'))

gap_exploration_oracle.to_csv(gap_exploration_trace_dir / 'bp_trajectory_oracle.csv', index=False)
oracle_summary.to_csv(gap_exploration_data_paths.output('bp_trajectory_oracle_summary_csv'), index=False)
display(gap_metrics.head(), order_summary.head(), snapshot_summary.head())


## Experiment 6: practical BP-only stopping rules

Test stopping rules that use available BP information. Compare them with the offline successful-snapshot opportunity, keeping the first syndrome-consistent endpoint distinct from a maximum-length diagnostic trace.


In [ ]:
stopping_paths = [gap_exploration_trace_dir / 'bp_stopping_trials.csv', gap_exploration_data_paths.output('bp_stopping_rules_csv')]
existing_stopping = sum((path.is_file() for path in stopping_paths))
stopping_cache_ready = gap_exploration_config['reuse_existing_data'] and existing_stopping == len(stopping_paths)

if stopping_cache_ready:
    stopping_trials = read_experiment_csv(stopping_paths[0])
    stopping_trials = stopping_trials[stopping_trials.shot_id.isin(shots.shot_id)].copy()
    stopping_summary = filter_distances(read_experiment_csv(stopping_paths[1]))
    gap_exploration_log(2, {'stage': 'stopping_rules', 'source': 'cache'})

development_first_valid_margins = []
for shot in [] if stopping_cache_ready else shots[shots.split.eq('development')].itertuples(index=False):
    gap_exploration_context, gap_exploration_syndrome, gap_exploration_truth, gap_exploration_trace = case_from_row(shot)
    if gap_exploration_trace.first_valid is not None:
        development_first_valid_margins.append(float(np.min(np.abs(gap_exploration_trace.llr[gap_exploration_trace.first_valid]))))

if not stopping_cache_ready:
    assert development_first_valid_margins
margin_thresholds = [] if stopping_cache_ready else np.unique(np.quantile(development_first_valid_margins, gap_exploration_config['trigger_quantiles']))
stopping_rows = []

for shot in tqdm([] if stopping_cache_ready else shots.itertuples(index=False), total=0 if stopping_cache_ready else len(shots), desc='BP stopping rules', disable=gap_exploration_config['print_level'] == 1 or stopping_cache_ready):
    gap_exploration_context, gap_exploration_syndrome, gap_exploration_truth, gap_exploration_trace = case_from_row(shot)
    first = gap_exploration_trace.first_valid
    if first is None:
        stopping_rows.append({
            'shot_id': shot.shot_id,
            'distance': shot.distance,
            'p': shot.p,
            'split': shot.split,
            'rule': 'S0_first_valid',
            'parameter': 0,
            'iterations': gap_exploration_config['bp_max_iterations'],
            'logical_correct': 0,
        })

        for threshold in margin_thresholds:
            stopping_rows.append({
                'shot_id': shot.shot_id,
                'distance': shot.distance,
                'p': shot.p,
                'split': shot.split,
                'rule': 'S4_margin',
                'parameter': float(threshold),
                'iterations': gap_exploration_config['long_bp_iterations'],
                'logical_correct': 0,
            })

        for patience in gap_exploration_config['patience_values']:
            for gap_exploration_rule in ['S1_last_valid', 'S1_lowest_prior_cost', 'S1_most_frequent', 'S2_logical_persistence', 'S3_valid_window']:
                stopping_rows.append({
                    'shot_id': shot.shot_id,
                    'distance': shot.distance,
                    'p': shot.p,
                    'split': shot.split,
                    'rule': gap_exploration_rule,
                    'parameter': patience,
                    'iterations': gap_exploration_config['long_bp_iterations'],
                    'logical_correct': 0,
                })

        continue

    stopping_rows.append({
        'shot_id': shot.shot_id,
        'distance': shot.distance,
        'p': shot.p,
        'split': shot.split,
        'rule': 'S0_first_valid',
        'parameter': 0,
        'iterations': first,
        'logical_correct': int(logical_correct(gap_exploration_context, gap_exploration_trace.hard[first], gap_exploration_truth)),
    })

    for threshold in margin_thresholds:
        qualified = [t for t in range(first, gap_exploration_config['long_bp_iterations'] + 1) if gap_exploration_trace.valid[t] and np.min(np.abs(gap_exploration_trace.llr[t])) >= threshold]
        valid_until_end = [t for t in range(first, gap_exploration_config['long_bp_iterations'] + 1) if gap_exploration_trace.valid[t]]
        gap_exploration_chosen = qualified[0] if qualified else valid_until_end[-1]
        stopping_rows.append({
            'shot_id': shot.shot_id,
            'distance': shot.distance,
            'p': shot.p,
            'split': shot.split,
            'rule': 'S4_margin',
            'parameter': float(threshold),
            'iterations': gap_exploration_chosen,
            'logical_correct': int(logical_correct(gap_exploration_context, gap_exploration_trace.hard[gap_exploration_chosen], gap_exploration_truth)),
        })

    for patience in gap_exploration_config['patience_values']:
        end = min(first + patience, gap_exploration_config['long_bp_iterations'])
        gap_exploration_indices = [t for t in range(first, end + 1) if gap_exploration_trace.valid[t]]
        gap_exploration_candidates = [gap_exploration_trace.hard[t] for t in gap_exploration_indices]
        assert gap_exploration_candidates
        choices = {
            'S1_last_valid': (gap_exploration_candidates[-1], end),
            'S1_lowest_prior_cost': (min(gap_exploration_candidates, key=lambda e: gap_exploration_prior_cost(gap_exploration_context, e)), end),
        }

        keys = [e.tobytes() for e in gap_exploration_candidates]
        mode = max(set(keys), key=keys.count)
        choices['S1_most_frequent'] = (gap_exploration_candidates[keys.index(mode)], end)
        sectors = [gap_exploration_bits(gap_exploration_context.L @ e % 2) for e in gap_exploration_candidates]
        persistent_index = next((i + patience - 1 for i in range(len(gap_exploration_candidates) - patience + 1) if len(set(sectors[i:i + patience])) == 1), len(gap_exploration_candidates) - 1)
        choices['S2_logical_persistence'] = (gap_exploration_candidates[persistent_index], gap_exploration_indices[persistent_index])
        choices['S3_valid_window'] = choices['S1_lowest_prior_cost']

        for gap_exploration_rule, (corr, stop_iteration) in choices.items():
            stopping_rows.append({
                'shot_id': shot.shot_id,
                'distance': shot.distance,
                'p': shot.p,
                'split': shot.split,
                'rule': gap_exploration_rule,
                'parameter': patience,
                'iterations': stop_iteration,
                'logical_correct': int(logical_correct(gap_exploration_context, corr, gap_exploration_truth)),
            })

if not stopping_cache_ready:
    stopping_trials = pd.DataFrame(stopping_rows).merge(shots[['shot_id', 'bposd_logical_correct', 'tesseract_logical_correct']], on='shot_id', how='left')
    stopping_trials['false_stop'] = 1 - stopping_trials.logical_correct
    stopping_trials['gap_repaired'] = ((stopping_trials.bposd_logical_correct == 0) & (stopping_trials.logical_correct == 1) & (stopping_trials.tesseract_logical_correct == 1)).astype(int)
    stopping_test = stopping_trials[stopping_trials.split.eq('test')]
    stopping_summary = stopping_test.groupby(['distance', 'p', 'rule', 'parameter'], as_index=False).agg(
        shots=('shot_id', 'size'),
        LER=('logical_correct', lambda x: 1 - x.mean()),
        mean_iterations=('iterations', 'mean'),
        p95_iterations=('iterations', lambda x: x.quantile(0.95)),
        false_stop_rate=('false_stop', 'mean'),
        gap_repairs=('gap_repaired', 'sum'),
    )
    gap_denominator = shots[shots.split.eq('test')].groupby(['distance', 'p']).gap_target.sum()
    stopping_summary['gap_closure_fraction'] = stopping_summary.apply(lambda r: r.gap_repairs / gap_denominator[r.distance, r.p] if gap_denominator[r.distance, r.p] else np.nan, axis=1)

    stopping_trials.to_csv(stopping_paths[0], index=False)
    stopping_summary.to_csv(stopping_paths[1], index=False)
    gap_exploration_log(2, {'stage': 'stopping_rules', 'source': 'experiment'})

stopping_summary.head()


## Experiments 7–10: false stops, selective OSD, OSD-0 audit, and BP-soft value

Audit false acceptance and the OSD-0 reference, then evaluate selective OSD and alternative soft inputs. Track repaired failures and newly introduced failures together with the OSD invocation rate.


In [ ]:
selective_paths = [gap_exploration_data_paths.output('false_stop_models_csv'), gap_exploration_data_paths.output('selective_osd_frontier_csv'), gap_exploration_data_paths.output('osd0_audit_csv'), gap_exploration_data_paths.output('osd0_escalation_csv'), gap_exploration_data_paths.output('bp_soft_information_value_csv'), gap_exploration_data_paths.output('experiment_status_csv'), gap_exploration_trace_dir / 'false_stop_scores.csv']
selective_cache_ready = gap_exploration_config['reuse_existing_data'] and all((path.is_file() for path in selective_paths))
if selective_cache_ready:
    false_stop_models = read_experiment_csv(selective_paths[0])
    selective_frontier = read_experiment_csv(selective_paths[1])
    audit_summary = read_experiment_csv(selective_paths[2])
    audit_escalation = read_experiment_csv(selective_paths[3])
    soft_value = read_experiment_csv(selective_paths[4])
    gap_exploration_experiment_status = read_experiment_csv(selective_paths[5])
    cached_scores = read_experiment_csv(selective_paths[6])
    gap_exploration_log(2, {'stage': 'experiments_7_10', 'source': 'cache'})

FEATURES = ['bp_iterations', 'llr_min', 'llr_q10', 'llr_q25', 'llr_median', 'llr_gap_5_6', 'recent_valid_count', 'recent_unique_valid', 'recent_logical_switches', 'belief_instability']
valid_cases = shots[shots.bp_syndrome_consistent.eq(1)].copy()
gap_exploration_development = valid_cases[valid_cases.split.eq('development')]
test = valid_cases[valid_cases.split.eq('test')]

if selective_cache_ready:
    valid_cases = valid_cases.merge(cached_scores, on='shot_id', how='left')
    gap_exploration_development = valid_cases[valid_cases.split.eq('development')]
    test = valid_cases[valid_cases.split.eq('test')]
class_counts = gap_exploration_development.false_stop.value_counts()
model_available = len(class_counts) == 2 and int(class_counts.min()) >= 2
models = {
    'logistic': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced')),
    'tree': DecisionTreeClassifier(
        max_depth=3,
        min_samples_leaf=20,
        class_weight='balanced',
        random_state=gap_exploration_config['random_seed'],
    ),
}
model_rows = []
test_scores = {}

for gap_exploration_name, model in models.items() if model_available and (not selective_cache_ready) else []:
    groups = gap_exploration_development.shot_id
    fold_count = min(gap_exploration_config['cv_folds'], int(class_counts.min()))
    folds = StratifiedGroupKFold(n_splits=fold_count, shuffle=True, random_state=gap_exploration_config['random_seed'])

    oof = cross_val_predict(model, gap_exploration_development[FEATURES], gap_exploration_development.false_stop, groups=groups, cv=folds, method='predict_proba')[:, 1]
    model.fit(gap_exploration_development[FEATURES], gap_exploration_development.false_stop)
    score = model.predict_proba(test[FEATURES])[:, 1]
    test_scores[gap_exploration_name] = score
    test_auc = roc_auc_score(test.false_stop, score) if test.false_stop.nunique() == 2 else np.nan
    model_rows.append({
        'model': gap_exploration_name,
        'development_AUROC': roc_auc_score(gap_exploration_development.false_stop, oof),
        'test_AUROC': test_auc,
    })

if not model_available and (not selective_cache_ready):
    model_rows = [{
        'model': name,
        'status': 'insufficient_events',
        'development_AUROC': np.nan,
        'test_AUROC': np.nan,
    } for name in models]
elif not selective_cache_ready:
    for gap_exploration_row in model_rows:
        gap_exploration_row['status'] = 'trained'

if not selective_cache_ready:
    false_stop_models = pd.DataFrame(model_rows)
    false_stop_models.to_csv(selective_paths[0], index=False)

if not selective_cache_ready:
    valid_cases['false_stop_risk'] = np.nan
if model_available and (not selective_cache_ready):
    primary = models['logistic']
    valid_cases['false_stop_risk'] = primary.predict_proba(valid_cases[FEATURES])[:, 1]
gap_exploration_development = valid_cases[valid_cases.split.eq('development')]
test = valid_cases[valid_cases.split.eq('test')]
gap_exploration_thresholds = np.quantile(valid_cases.loc[valid_cases.split.eq('development'), 'false_stop_risk'], gap_exploration_config['trigger_quantiles']) if model_available and (not selective_cache_ready) else []
frontier = []

for q, threshold in zip(gap_exploration_config['trigger_quantiles'], gap_exploration_thresholds):
    for gap_exploration_split_name, evaluated in [('development', gap_exploration_development), ('test', test)]:
        trigger = evaluated.false_stop_risk >= threshold
        gap_exploration_success = np.where(trigger, evaluated.forced_osd_logical_correct, evaluated.bposd_logical_correct)
        valid_gap = np.sum((evaluated.bposd_logical_correct == 0) & (evaluated.tesseract_logical_correct == 1))
        total_gap = shots.loc[shots.split.eq(gap_exploration_split_name), 'gap_target'].sum()
        closed = np.sum((evaluated.bposd_logical_correct == 0) & (gap_exploration_success == 1) & (evaluated.tesseract_logical_correct == 1))
        frontier.append({
            'split': gap_exploration_split_name,
            'method': 'logistic',
            'quantile': q,
            'threshold': threshold,
            'OSD_invocation_rate': trigger.mean(),
            'valid_gap_closure_fraction': closed / valid_gap if valid_gap else np.nan,
            'gap_closure_fraction': closed / total_gap if total_gap else np.nan,
            'LER_valid_BP_subset': 1 - gap_exploration_success.mean(),
        })

for feature, direction in [] if selective_cache_ready else [('bp_iterations', 'high'), ('llr_min', 'low'), ('recent_logical_switches', 'high')]:
    for q in gap_exploration_config['trigger_quantiles']:
        threshold = gap_exploration_development[feature].quantile(q if direction == 'high' else 1 - q)
        for gap_exploration_split_name, evaluated in [('development', gap_exploration_development), ('test', test)]:
            trigger = evaluated[feature].ge(threshold) if direction == 'high' else evaluated[feature].le(threshold)
            gap_exploration_success = np.where(trigger, evaluated.forced_osd_logical_correct, evaluated.bposd_logical_correct)
            valid_gap = np.sum(evaluated.gap_target)
            total_gap = shots.loc[shots.split.eq(gap_exploration_split_name), 'gap_target'].sum()
            closed = np.sum((evaluated.gap_target == 1) & (gap_exploration_success == 1))
            frontier.append({
                'split': gap_exploration_split_name,
                'method': feature,
                'quantile': q,
                'threshold': threshold,
                'OSD_invocation_rate': trigger.mean(),
                'valid_gap_closure_fraction': closed / valid_gap if valid_gap else np.nan,
                'gap_closure_fraction': closed / total_gap if total_gap else np.nan,
                'LER_valid_BP_subset': 1 - gap_exploration_success.mean(),
            })

if not selective_cache_ready:
    selective_frontier = pd.DataFrame(frontier)
    selective_frontier.to_csv(selective_paths[1], index=False)
audit = valid_cases.assign(osd0_disagreement=lambda x: x.osd0_disagrees.astype(int), logical_disagreement=lambda x: x.osd0_logical_disagrees.astype(int))
audit_summary = pd.DataFrame([{
    'audit': 'candidate_disagreement',
    'trigger_rate': audit.osd0_disagreement.mean(),
    'false_stop_recall': audit.loc[audit.false_stop.eq(1), 'osd0_disagreement'].mean(),
}, {
    'audit': 'logical_disagreement',
    'trigger_rate': audit.logical_disagreement.mean(),
    'false_stop_recall': audit.loc[audit.false_stop.eq(1), 'logical_disagreement'].mean(),
}, {
    'audit': 'positive_cost_gain',
    'trigger_rate': audit.osd0_cost_gain.gt(0).mean(),
    'false_stop_recall': audit.loc[audit.false_stop.eq(1), 'osd0_cost_gain'].gt(0).mean(),
}])
audit_test = audit[audit.split.eq('test')]
escalation_rows = []

for gap_exploration_name, trigger in {
    'candidate_disagreement': audit_test.osd0_disagreement.eq(1),
    'logical_disagreement': audit_test.logical_disagreement.eq(1),
    'positive_cost_gain': audit_test.osd0_cost_gain.gt(0),
}.items():
    gap_exploration_success = np.where(trigger, audit_test.forced_osd_logical_correct, audit_test.bposd_logical_correct)
    valid_gap = np.sum(audit_test.gap_target)
    total_gap = shots.loc[shots.split.eq('test'), 'gap_target'].sum()
    closed = np.sum((audit_test.gap_target == 1) & (gap_exploration_success == 1))
    escalation_rows.append({
        'audit': gap_exploration_name,
        'full_osd_invocation_rate': trigger.mean(),
        'LER_valid_BP_subset': 1 - gap_exploration_success.mean(),
        'valid_gap_closure_fraction': closed / valid_gap if valid_gap else np.nan,
        'gap_closure_fraction': closed / total_gap if total_gap else np.nan,
    })

audit_escalation = pd.DataFrame(escalation_rows)
audit_summary.to_csv(gap_exploration_data_paths.output('osd0_audit_csv'), index=False)
audit_escalation.to_csv(gap_exploration_data_paths.output('osd0_escalation_csv'), index=False)
soft_subset = shots[shots.bp_logical_correct.eq(0) & shots.forced_osd_logical_correct.eq(1)]
soft_value = pd.DataFrame([{
    'subset_cases': len(soft_subset),
    'prior_osd_success': soft_subset.prior_osd_correct.mean(),
    'bp_posterior_osd_success': soft_subset.osdcs_correct.mean(),
    'early_snapshot_osd_success': soft_subset.early_osd_correct.mean(),
    'delta_R_BP_soft': soft_subset.osdcs_correct.mean() - soft_subset.prior_osd_correct.mean(),
}])
gap_exploration_experiment_status = pd.DataFrame([{
    'stage': 'experiments_7_10',
    'development_cases': len(gap_exploration_development),
    'development_false_stops': int(gap_exploration_development.false_stop.sum()),
    'test_false_stops': int(test.false_stop.sum()),
    'development_gap_cases': int(shots.loc[shots.split.eq('development'), 'gap_target'].sum()),
    'classifier_status': 'trained' if model_available else 'insufficient_events',
}])

soft_value.to_csv(gap_exploration_data_paths.output('bp_soft_information_value_csv'), index=False)
gap_exploration_experiment_status.to_csv(gap_exploration_data_paths.output('experiment_status_csv'), index=False)
valid_cases[['shot_id', 'false_stop_risk']].to_csv(gap_exploration_trace_dir / 'false_stop_scores.csv', index=False)
display(false_stop_models, selective_frontier, audit_summary, soft_value, gap_exploration_experiment_status)


## Experiments 11–12: optimize BP for OSD and instability-aware handoff

Scan BP settings as soft-information generators for OSD and test instability-aware handoff. Better stand-alone BP convergence is not assumed to imply better OSD ordering.


In [ ]:
handoff_paths = [gap_exploration_trace_dir / 'bp_handoff_grid_trials.csv', gap_exploration_data_paths.output('bp_handoff_optimization_csv'), gap_exploration_data_paths.output('instability_aware_handoff_csv')]
existing_handoff = sum((path.is_file() for path in handoff_paths))
handoff_cache_ready = gap_exploration_config['reuse_existing_data'] and existing_handoff == len(handoff_paths)

if handoff_cache_ready:
    handoff_trials = read_experiment_csv(handoff_paths[0])
    handoff_trials = handoff_trials[handoff_trials.shot_id.isin(shots.shot_id)].copy()
    handoff_summary = read_experiment_csv(handoff_paths[1])
    instability_branch = filter_distances(read_experiment_csv(handoff_paths[2]))
    gap_exploration_log(2, {'stage': 'handoff_grid', 'source': 'cache'})

development_grid = shots[shots.split.eq('development')].copy()
development_grid['_stratum_rank'] = development_grid.groupby(['distance', 'p']).cumcount()
grid_cases = development_grid.sort_values(['_stratum_rank', 'distance', 'p']).head(gap_exploration_config['handoff_grid_cases'])
gap_exploration_grid_rows = []

for gap_exploration_scaling in tqdm([] if handoff_cache_ready else gap_exploration_config['scaling_values'], desc='BP handoff grid', disable=gap_exploration_config['print_level'] == 1 or handoff_cache_ready):
    for gap_exploration_damping in gap_exploration_config['damping_values']:
        for gap_exploration_schedule in gap_exploration_config['schedule_values']:
            if gap_exploration_schedule == 'serial' and gap_exploration_damping != 0:
                continue
            for gap_exploration_iterations in gap_exploration_config['iteration_values']:
                for gap_exploration_row in grid_cases.itertuples(index=False):
                    gap_exploration_context, gap_exploration_syndrome, gap_exploration_truth, gap_exploration_ = case_from_row(gap_exploration_row, iterations=1)
                    started = perf_counter()
                    gap_exploration_trace = bp_trace(gap_exploration_context, gap_exploration_syndrome, gap_exploration_iterations, gap_exploration_scaling, gap_exploration_damping, schedule=gap_exploration_schedule)
                    gap_exploration_llr = gap_exploration_trace.llr[-1]
                    gap_exploration_hard = gap_exploration_trace.hard[-1]
                    osd = osd_snapshot(gap_exploration_context, gap_exploration_syndrome, gap_exploration_llr, primary_order_for(gap_exploration_context))
                    gap_exploration_grid_rows.append({
                        'shot_id': gap_exploration_row.shot_id,
                        'scaling': gap_exploration_scaling,
                        'damping': gap_exploration_damping,
                        'schedule': gap_exploration_schedule,
                        'iterations': gap_exploration_iterations,
                        'bp_correct': int(logical_correct(gap_exploration_context, gap_exploration_hard, gap_exploration_truth)),
                        'forced_osd_correct': int(logical_correct(gap_exploration_context, osd, gap_exploration_truth)),
                        'osd0_correct': int(logical_correct(gap_exploration_context, osd_snapshot(gap_exploration_context, gap_exploration_syndrome, gap_exploration_llr, 0), gap_exploration_truth)),
                        'runtime_seconds': perf_counter() - started,
                    })

if not handoff_cache_ready:
    handoff_trials = pd.DataFrame(gap_exploration_grid_rows)
    handoff_summary = handoff_trials.groupby(['scaling', 'damping', 'schedule', 'iterations'], as_index=False).agg(
        cases=('shot_id', 'size'),
        BP_LER=('bp_correct', lambda x: 1 - x.mean()),
        FOSD_LER=('forced_osd_correct', lambda x: 1 - x.mean()),
        OSD0_success=('osd0_correct', 'mean'),
        mean_runtime_seconds=('runtime_seconds', 'mean'),
    )
    instability_branch = shots.groupby(['distance', 'p'], as_index=False).agg(
        syndrome_invalid_rate=('bp_syndrome_consistent', lambda x: 1 - x.mean()),
        belief_instability=('belief_instability', 'mean'),
        mean_adaptive_damping=('instability_damping', 'mean'),
        instability_branch_success=('instability_osd_correct', 'mean'),
        mean_branch_runtime_seconds=('instability_branch_runtime_seconds', lambda x: x[x > 0].mean()),
    )

    handoff_trials.to_csv(handoff_paths[0], index=False)
    handoff_summary.to_csv(handoff_paths[1], index=False)
    instability_branch.to_csv(handoff_paths[2], index=False)
    gap_exploration_log(2, {'stage': 'handoff_grid', 'source': 'experiment'})

display(handoff_summary.head(), instability_branch.head())


## Final proposed decoders and paired accuracy–cost benchmark

Apply the proposed decoder rules to the paired benchmark and compare accuracy with cost. Oracle snapshot results are ceilings; practical rules must choose without logical truth or future observations.


In [ ]:
final_paths = [gap_exploration_trace_dir / 'final_decoder_trials.csv', gap_exploration_data_paths.output('final_benchmark_csv'), gap_exploration_data_paths.output('selected_selective_policy_csv')]
existing_final = sum((path.is_file() for path in final_paths))
final_cache_ready = gap_exploration_config['reuse_existing_data'] and existing_final == len(final_paths)

if final_cache_ready:
    gap_exploration_final_trials = read_experiment_csv(final_paths[0])
    gap_exploration_final_trials = gap_exploration_final_trials[gap_exploration_final_trials.shot_id.isin(shots.shot_id)].copy()
    final_benchmark = filter_distances(read_experiment_csv(final_paths[1]))
    selected_policy = read_experiment_csv(final_paths[2])
    gap_exploration_log(2, {'stage': 'final_benchmark', 'source': 'cache'})
else:
    development_logistic = selective_frontier[selective_frontier.method.eq('logistic') & selective_frontier.split.eq('development')].dropna(subset=['gap_closure_fraction'])
    development_rules = selective_frontier[selective_frontier.method.ne('logistic') & selective_frontier.split.eq('development')].dropna(subset=['gap_closure_fraction'])
    gap_exploration_candidates = development_logistic if len(development_logistic) else development_rules

    if len(gap_exploration_candidates):
        gap_exploration_chosen = gap_exploration_candidates.sort_values(['gap_closure_fraction', 'OSD_invocation_rate'], ascending=[False, True]).iloc[0]
        selected_method = str(gap_exploration_chosen.method)
        threshold = float(gap_exploration_chosen.threshold)
        selection_status = 'selected_on_development'
    else:
        selected_method = 'no_selective_osd'
        threshold = np.inf
        selection_status = 'no_observed_development_gap'

    selected_policy = pd.DataFrame([{'method': selected_method, 'threshold': threshold, 'status': selection_status}])
    selected_policy.to_csv(final_paths[2], index=False)

scored = shots.merge(valid_cases[['shot_id', 'false_stop_risk']], on='shot_id', how='left')
selected_method = str(selected_policy.iloc[0].method)
threshold = float(selected_policy.iloc[0].threshold)
final_rows = []

for gap_exploration_row in [] if final_cache_ready else scored[scored.split.eq('test')].itertuples(index=False):
    risk = -np.inf if pd.isna(gap_exploration_row.false_stop_risk) else gap_exploration_row.false_stop_risk
    if selected_method == 'logistic':
        selective_call = bool(gap_exploration_row.bp_syndrome_consistent and risk >= threshold)
    elif selected_method == 'bp_iterations':
        selective_call = bool(gap_exploration_row.bp_syndrome_consistent and gap_exploration_row.bp_iterations >= threshold)
    elif selected_method == 'llr_min':
        selective_call = bool(gap_exploration_row.bp_syndrome_consistent and gap_exploration_row.llr_min <= threshold)
    elif selected_method == 'recent_logical_switches':
        selective_call = bool(gap_exploration_row.bp_syndrome_consistent and gap_exploration_row.recent_logical_switches >= threshold)
    else:
        selective_call = False

    audit_escalate = bool(gap_exploration_row.bp_syndrome_consistent and gap_exploration_row.osd0_logical_disagrees)
    gap_exploration_results = {
        'plain_BP': gap_exploration_row.bp_logical_correct,
        'standard_BPOSD': gap_exploration_row.bposd_logical_correct,
        'forced_OSD_BPOSD': gap_exploration_row.forced_osd_logical_correct,
        'Tesseract': gap_exploration_row.tesseract_logical_correct,
        'always_audit_BPOSD': gap_exploration_row.forced_osd_logical_correct if not gap_exploration_row.bp_syndrome_consistent or audit_escalate else gap_exploration_row.osd0_correct,
        'selective_OSD_BPOSD': gap_exploration_row.forced_osd_logical_correct if not gap_exploration_row.bp_syndrome_consistent or selective_call else gap_exploration_row.bposd_logical_correct,
        'failure_aware_BPOSD': gap_exploration_row.instability_osd_correct if not gap_exploration_row.bp_syndrome_consistent else gap_exploration_row.forced_osd_logical_correct if selective_call else gap_exploration_row.bposd_logical_correct,
    }

    gap_exploration_costs = {
        'plain_BP': (gap_exploration_row.plain_runtime_seconds, 0, 0),
        'standard_BPOSD': (gap_exploration_row.standard_runtime_seconds, 0, int(not gap_exploration_row.bp_syndrome_consistent)),
        'forced_OSD_BPOSD': (gap_exploration_row.forced_runtime_seconds, 0, 1),
        'Tesseract': (gap_exploration_row.tesseract_runtime_seconds, 0, 0),
        'always_audit_BPOSD': (gap_exploration_row.standard_runtime_seconds + (gap_exploration_row.osd0_runtime_seconds if gap_exploration_row.bp_syndrome_consistent else 0) + (gap_exploration_row.osdcs_runtime_seconds if audit_escalate else 0), int(gap_exploration_row.bp_syndrome_consistent), int(not gap_exploration_row.bp_syndrome_consistent or audit_escalate)),
        'selective_OSD_BPOSD': (gap_exploration_row.standard_runtime_seconds + (gap_exploration_row.osdcs_runtime_seconds if selective_call else 0), 0, int(not gap_exploration_row.bp_syndrome_consistent or selective_call)),
        'failure_aware_BPOSD': (gap_exploration_row.instability_branch_runtime_seconds if not gap_exploration_row.bp_syndrome_consistent else gap_exploration_row.standard_runtime_seconds + (gap_exploration_row.osdcs_runtime_seconds if selective_call else 0), 0, int(not gap_exploration_row.bp_syndrome_consistent or selective_call)),
    }

    for decoder, gap_exploration_correct in gap_exploration_results.items():
        gap_exploration_runtime, osd0_call, full_call = gap_exploration_costs[decoder]
        final_rows.append({
            'shot_id': gap_exploration_row.shot_id,
            'distance': gap_exploration_row.distance,
            'p': gap_exploration_row.p,
            'decoder': decoder,
            'logical_correct': int(gap_exploration_correct),
            'osd_invoked': int(osd0_call or full_call),
            'osd0_invoked': osd0_call,
            'osd1_invoked': 0,
            'full_osd_invoked': full_call,
            'runtime_seconds': gap_exploration_runtime,
            'bp_iterations': gap_exploration_row.bp_iterations,
        })

if not final_cache_ready:
    gap_exploration_final_trials = pd.DataFrame(final_rows)
    gap_exploration_final_trials.to_csv(final_paths[0], index=False)
    final_benchmark = gap_exploration_final_trials.groupby(['distance', 'p', 'decoder'], as_index=False).agg(
        shots=('shot_id', 'size'),
        logical_errors=('logical_correct', lambda x: np.sum(1 - x)),
        OSD_invocation_rate=('osd_invoked', 'mean'),
        OSD0_fraction=('osd0_invoked', 'mean'),
        OSD1_fraction=('osd1_invoked', 'mean'),
        full_OSD_fraction=('full_osd_invoked', 'mean'),
        mean_BP_iterations=('bp_iterations', 'mean'),
        mean_runtime_seconds=('runtime_seconds', 'mean'),
        median_runtime_seconds=('runtime_seconds', 'median'),
        p95_runtime_seconds=('runtime_seconds', lambda x: x.quantile(0.95)),
    )
    final_benchmark['LER'] = final_benchmark.logical_errors / final_benchmark.shots
    final_benchmark[['LER_CI95_low', 'LER_CI95_high']] = final_benchmark.apply(lambda row: pd.Series(wilson_interval(int(row.logical_errors), int(row.shots))), axis=1)
    gap_exploration_reference = final_benchmark.pivot(index=['distance', 'p'], columns='decoder', values='LER')

    def closure(row):
        b = gap_exploration_reference.loc[(row.distance, row.p), 'standard_BPOSD']
        t = gap_exploration_reference.loc[(row.distance, row.p), 'Tesseract']
        return (b - row.LER) / (b - t) if not np.isclose(b, t) else np.nan

    final_benchmark['gap_closure_fraction'] = final_benchmark.apply(closure, axis=1)
    final_benchmark.to_csv(final_paths[1], index=False)
    gap_exploration_log(2, {'stage': 'final_benchmark', 'source': 'analysis', 'policy': selected_method})

final_benchmark.head()


In [ ]:
fig, gap_exploration_axes = plt.subplots(2, 2, figsize=(13, 9))
gap_plot = gap_summary.groupby(['p', 'gap_class']).shots.sum().unstack(fill_value=0)
gap_plot.plot(kind='bar', stacked=True, ax=gap_exploration_axes[0, 0])
gap_exploration_axes[0, 0].set_title('BPOSD gap decomposition')
sns.lineplot(
    data=gap_metrics,
    x='p',
    y='eta_stop',
    hue='distance',
    marker='o',
    ax=gap_exploration_axes[0, 1],
)
gap_exploration_axes[0, 1].set_title('Early-stop gap closure')
sns.lineplot(
    data=snapshot_summary,
    x='iteration',
    y='osd_success_rate',
    hue='distance',
    ax=gap_exploration_axes[1, 0],
)
gap_exploration_axes[1, 0].set_title('OSD success by BP snapshot')
sns.lineplot(
    data=selective_frontier[selective_frontier.split.eq('test')],
    x='OSD_invocation_rate',
    y='gap_closure_fraction',
    hue='method',
    marker='o',
    ax=gap_exploration_axes[1, 1],
)
gap_exploration_axes[1, 1].set_title('Selective OSD frontier (held-out test)')
plt.tight_layout()

plt.show()
fig, gap_exploration_axes = plt.subplots(1, 2, figsize=(12, 4))
for (gap_exploration_distance, decoder), gap_exploration_group in final_benchmark.groupby(['distance', 'decoder']):
    gap_exploration_group = gap_exploration_group.sort_values('p')
    lower = np.maximum(0, gap_exploration_group.LER - gap_exploration_group.LER_CI95_low)
    upper = np.maximum(0, gap_exploration_group.LER_CI95_high - gap_exploration_group.LER)
    gap_exploration_axes[0].errorbar(
        gap_exploration_group.p,
        gap_exploration_group.LER,
        yerr=np.vstack([lower, upper]),
        marker='o',
        capsize=2,
        label=f'd={gap_exploration_distance} {decoder}',
    )

gap_exploration_axes[0].set_yscale('log')
gap_exploration_axes[0].set_xlabel('p')
gap_exploration_axes[0].set_ylabel('LER')
gap_exploration_axes[0].legend(fontsize=7)
sns.scatterplot(
    data=final_benchmark,
    x='OSD_invocation_rate',
    y='gap_closure_fraction',
    hue='decoder',
    style='distance',
    ax=gap_exploration_axes[1],
)
plt.tight_layout()

plt.show()
assert shots.shot_id.is_unique and set(shots.split) == {'development', 'test'}
assert shots[['bposd_logical_correct', 'forced_osd_logical_correct', 'tesseract_logical_correct']].isin([0, 1]).all().all()
assert snapshots.groupby(['shot_id', 'iteration']).size().eq(1).all()
assert set(final_benchmark.decoder) == {'plain_BP', 'standard_BPOSD', 'forced_OSD_BPOSD', 'always_audit_BPOSD', 'selective_OSD_BPOSD', 'failure_aware_BPOSD', 'Tesseract'}
gap_exploration_log(2, {
    'shots': len(shots),
    'stats_files': len(list(gap_exploration_stats.glob('*.csv'))),
    'trace_files': len(list(gap_exploration_trace_dir.glob('*.csv'))),
})
